# MandiaCompact S2S — données renforcées et comparaison contrôlée
Poids et tokenizer initialisés par Mandia, sans modèle préentraîné. Les checkpoints complets restent dans Drive. Cette nouvelle architecture est incompatible avec les checkpoints v0, qui sont conservés.

**Objectif de cette expérience :** apprendre la traduction français→baoulé et trois intentions de réponse (saluer, répondre à « comment vas-tu », prendre congé). Un corpus de traductions n'est pas un corpus général de conversations.

Hybride : encodeur français bidirectionnel à 2 blocs, décodeur à 6 blocs, 4 convolutions causales + 2 GQA. Chaque bloc consulte la source par attention croisée. Référence : attention GQA partout, paramètres ajustés à ±0,2%. RMSNorm, RoPE, SwiGLU, partage des embeddings et cache compact sont des briques connues.

**Protocole fixé avant exécution :** graine 42, 6000 mises à jour par modèle, 500 étapes de débruitage baoulé puis 5500 étapes à objectif 50% traduction directe / 20% lexique / 10% réponses simples / 20% débruitage. Même suite de tirages, mêmes tokens réels et supervisés. Sélection par validation équilibrée entre les quatre domaines, sans sélection sur le test. Une seule graine ne suffit pas à une publication comparative.

Sources : [mtBCI-1.0](https://github.com/Kjeanclaude/mtBCI-1.0-Corpus), script latin, licence CC Attribution déclarée par l'auteur (version non spécifiée, provenance d'origine conservée) ; [Google SMOL/GATITOS](https://huggingface.co/datasets/google/smol), CC-BY-4.0, lexique français via pivot anglais ; [Google WAXAL](https://huggingface.co/datasets/google/WaxalNLP), transcriptions TTS baoulé, University of Ghana, CC-BY-4.0, split train officiel seulement. Aucun audio ne sera téléchargé par cette expérience.

Les chapitres bibliques de mtBCI sont groupés avant séparation ; les doublons exacts et collisions de holdout sont exclus. Les transcriptions audio baoulé servent au débruitage, pas à inventer des traductions françaises. La Bible 1998 et le NT 1974 sont identifiés chez les éditeurs ; leur intégration intégrale demandera une version et des conditions de réutilisation établies. Le dataset HF `abdouaziiz/bci-bible` est exclu : les échantillons inspectés paraissent mandingues, pas baoulé.


## Résultats vérifiés

# MandiaCompact S2S — comparaison terminée

Deux modèles construits et entraînés depuis zéro, 6000 mises à jour chacun, une graine (42), même corpus et même budget de tokens vérifié. Poids et checkpoints complets conservés dans Drive.

| Mesure | Hybride | Référence GQA |
|---|---:|---:|
| Paramètres | 13,782,528 | 13,796,352 |
| Meilleur pas de validation | 1,250 | 1,250 |
| Perte de validation équilibrée ↓ | 3.9591 | 4.0725 |
| Perte test équilibrée ↓ | 3.6496 | 3.7408 |
| Tokens/s L4 bf16 ↑ | 110.14 | 89.41 |
| Cache décodeur, octets ↓ | 271,360 | 390,144 |

| Traduction test, 32 exemples par domaine | chrF++ hybride | chrF++ référence |
|---|---:|---:|
| Quotidien | 26.83 | 23.98 |
| Logiciels | 18.67 | 22.30 |
| Extraits bibliques | 20.49 | 20.18 |
| Lexique via anglais | 4.96 | 5.67 |

Dans ce benchmark : ratio de débit hybride/référence 1.232 ; ratio de cache 0.696. Décodage de 64 tokens forcés, même source, cinq répétitions après chauffe ; mémoire encodeur et allocations runtime exclues du cache. Ces ratios ne prouvent pas une efficacité universelle.
Candidat choisi sur validation : `hybrid`. Tokens réels : 8,557,013 ; tokens supervisés : 4,174,897 par modèle.

Les 14 démonstrations apprises et six reformulations couvrent seulement trois intentions. Les erreurs de traduction restent nombreuses ; une évaluation par des locuteurs baoulé et plusieurs graines restent nécessaires. Aucun résultat ne démontre une nouveauté scientifique. Les transcriptions WAXAL servent au débruitage ; ce modèle ne traite pas encore la voix.

Sorties effectivement générées par l’hybride : Bonjour ! → Anouho ; Comment vas-tu ? → mi wun ti kpa ; Au revoir → ye tɛ wo nu. Les deux modèles retrouvent 14/14 références apprises ; les six reformulations manuelles donnent 6/6 pour l’hybride et 5/6 pour la référence. Elles ne constituent pas un test indépendant de conversation générale.
L’hybride est plus rapide dans ce test et utilise moins de cache. Il obtient une meilleure perte équilibrée, mais le chrF++ de la référence est meilleur sur les logiciels et le lexique. Une seule graine et 32 générations par domaine ne suffisent pas à établir une supériorité robuste.

Export CPU dynamique int8 : accepté par les critères fixés = True. Taille 18,658,043 octets contre 55,171,544 ; variation de perte de validation -0.0042. Les 14 sorties apprises restent correctes = True. Rechargement du state_dict vérifié. CPU FP32 147.28 tokens/s ; CPU int8 114.98 tokens/s. Matériel Colab uniquement, pas de garantie sur un autre processeur. Les couches embedding, convolution et normalisation restent FP32 ; API PyTorch dépréciée.
L’acceptation int8 porte sur la qualité et le format. L’export est moins rapide sur ce CPU ; il est conservé comme option de stockage, tandis que les poids FP32 restent disponibles.


In [ ]:
%pip -q install "tokenizers==0.22.1" "sacrebleu==2.5.1" pyarrow requests


## Données et Drive
Cette cellule contient les petits corpus préparés et leurs empreintes. Elle les décompresse directement dans Drive, sans télécharger les poids sur votre ordinateur. Code : Apache-2.0 ; données : licences originales, précisées par fichier dans les crédits et la provenance. Elle refuse d'écraser des données différentes d'un run existant.


In [ ]:
PREPARED_ASSETS={'parallel-prepare.jsonl': '...', 'replies-prepare.jsonl': 'H4sIAAAAAAAC/+XWvW7bMBAA4L1PcdUcobWdKUvhdghSoOgQ78LJvMhsSJ7AHwlqUKBbHiIP0BfoG3jKW4W0WgMJAk8JgcibcMejTh9OIG8KKYozKCy1aig3pBSXH4sTKBrLoa3GpLcoTSlI87gi5V2rpN8nU0SwTk//N0shj+76UaC1rNtd2Wc2PzhYeJ/CaFxPNoWXhsNmfEHMrqkSGDehXUnD3Cj64DQrOF+uLlbfL09AY9tK08CaTUfWoZdsQJCTjSEB9QDf0AiJu25Sp3Ftlb4kbuhtoBg2saajylInqSdR1UOlx5ozuELlaN9g9a8nMo2SbpN6Gj1+vbt5xnGW0XG6ivMcipeogp/yLC6yKU7X8DSH4RcOaw7THMU+RmuKrzp4xOxXvQyn1mQ8dOhKH+DTY1YtoQ8GvITrFt+OrgTUcCUNHRCe5RJehYQLtSRzRLzz3AOMStHPsuPgjkh5kUt5e4txio9I9jT3/G7/HIVwPdDBwy3mX0J1GSB2zvLJvX8g8Pd30DOY8HY0Y08iuTxnOcto+fTSNT3N+etrbn/vrgJ++9dPXXPx+ppfIxqBQulSmzjNv/4BaO8Gk4QSAAA=', 'PROVENANCE.json': 'H4sIAAAAAAAC/21U247TMBB9368Y5QlE282tabv7xFYCwQqEEO+RY08bs44d2U631YoP4jv4McbujZZ9i33mzBmfmcnLDUDCzaC9S+7ghU507ph9Gvq6V4xja5RAS1gxL0Z72LGuV1iHqIjkh3vser+jc3Y8bxn3tRh6JTnzGASyqjyA3Brnam5sP7jaWyY1HZSSThpd45arQaAIjCJ9lYHO1xumpGA+MF7lTon4K7ATa57/eR9u0O4E291GXbovq6PIGSIBQqr5NXAWDcTjc5xZ+Wdm8RLOsvwaP0rO08V/0F4yOyFr43pUJ8osvwYOhFl+eX9ZQ3p8m8ItNeKcL5+X19AFs5gurpl7wSKdn5x1ZrAc661ytWtZPq0IT9Js1jDRVKu8qRZ5iRzLki2mYlZUjK0wnecNLxYo8mKGVVEWTZqLIm1wlouynKdlElPT2KB2GBIul/DeeyubIdQGArkixwQ0O2CDb429B+pP6D9o44Fc4HIlUdzDoGl81U7qNexLhd6aDWqm6dOiJytCmEWHzPIWSISpUcyicM0UcEVAjF4ZC3EMwFi5lprAfmjicAfDYs1c+qN9yaMZmHNyBI+G5OAzMj1eKkbDCW/yNC/fjuCBmUH9+Q3fUdGGCPhGUkqhgiWNubEsan5hvKUq4QeV4VRMD565J3cHnX9Yfhpnk3QEwsi7LJ1Mszy7lT+5k34SRCZZNU2zfXFra4aenAjFxeWEDxY1b28pCVDnmUMH74CbsN8e4WMcJ+At6z25O4Jn6VuIezje7yGE34MZPJz2z8FxAWFlTQdx2ILmoaXdwaCwjslXQ4/TQjLQdLlBOM/fPWjyfXz6e4DDjmkvOfTkUd/aWGxoE25bNrhAVzuQAilo3/pD+WH3gaYlBksahx51iKJoamaMjcODwYK17ghyk+Tm118hd8xXGwUAAA==', 'RESSOURCES-BAOULE.json': 'H4sIAAAAAAAC/6VY3W4bNxa+z1MQBnoVj0f/P9GVIreuG9spECfdYlEIHM4ZiQ6HnAw5spWiQB9ibxb7BHvfN+hVX6NPsueQo5Ecy3LdBQJF1pCHh993fr4zP79g7EgsQXyEdM7d0St21Gl1BlG7FbW7R8f0tARrqlKAxYf/xB8Y+9l/4iPNc6AtuXs9O4/aJy2/wz+qSkVPls4V9lUcL6RbVsmJMHn85ga4FopXKcTNxmhmyqKy2/0fpU7JwDclaLGMcRnjSi40pMzBnWMSTVSp1At2ZmwBioklLxyUOyas464ir4+KEgpe4tbMlIw7V8qkcvgnXg14KZYMf+FqZ6coZeHBuOBO6gkraGW5AvbHf+I//rVdqKQAbT0Gsxmb1pal0SwFvCIeOWErdIp+0cYx9FTITNLPlU6hVGu6QVGaFWgEBdAjxyVe8sif8Mvxfrx/mP5jesFec1MpYNfX7x6FfVktFnhCxgUg9nHKHbfgbLwwZqEg/oHfcXV18f12fwkrSd6SkX7WS0eCd3pZJxmn7cEo6QD0e/2snUHCeZb1kl6rP9ohXRidyQXtTXg1d24Pn0i7NOwl0wjsCiLF9aLiC/CkTjxGiVSSftzhgwCSCBcZeK+lR9StmcnY2ZJr/ggf0esfo95uTG4DwmSZFEg5Eo9w06cOnFuGfpRchOjIjQO1nrDgdAkZUDSCbWg6rh+R36m51crw9CnyXssEWavJa4/Ho0fZu729PUlotc+bOo5s3G2NWlGSjEeRXnOdy69mg6/GLR3pdPPtvtUN9FmlFAunv2QX0jrQOxBXiZJ2GTAOi94ZISHAPPv9Nwcs/fPXf5+vjCyBxWyqlPQhS4vlpwrY7Hwf1D4myhzxNKXEYETQIZWUIxPmPSLm4wCjZ4MyQuaFKTGbPbCYGuhOihnjpELoKwsMrQYjlnGdNjnmS0QOGjehB+FGT9ERiLiCW3ZNm/zu9njYey4t7UGvGyVKu2GvJgaIE4iqTMHWmL/p/KBJvyTyf0e8KKK9RyTe70jDbeQ2fj/k/Oo6lMuXIVD/Lt/PJTYNKWGlQq/UOvDCJKZXIJovkGjrfPh7Bw9zdMpXvNykTLDgXX6UIoU1ogZU2th/iRMh00LGZ19fxd3H6lIAQhnBnSn3XhqjTWIZhe2l/U5sEVDm0oYoxIjEjiEkBmoNCXOGglRm679UHHxVmn5/fjAGM45NFeMFbLJeYh/DxHkQPk3zDiBECVl+ePudHLx36vbiJXyqMBAsPWcfYX3M8P5LZ6O6pQmWIdnYXJhXE5ZqOeNCGMxaxIphnNI2JkpA9J7IyW+/YTxJTcU/S/mZiAu3eXabO2xkt9dBbwCjFHqdJOukbYDBOEvG42wwHA2h2806WVckIunuBE5pbgmY9rA13sLF80LBvJYiZLc9+3bebs2nb96czlqjvY3qni7YBz3ckdSB9BWKHlpIvWnbsUINLAoMAHZJ0akXxz75QrpMGEalAu6lUg0M8zgfZGCa3pgq51qGkxT35zybgMZMvGtmnnlRNw8lbP6hvbVrEq+z0nkNb6fV6x6EpKn4sa9zqBSQ4zx0BapQCLSjq9eAM7IBtN7eL8le7U3fn55fR9PT796+v5xenZ/cWIyOgzideSHF3l2+vcCeeDa9Pr9++7flmM2NepibSwKQfa0XVK9JCE9YEMWskCvkWcGdFFiHcu7EEq0/Xw1tLIRSgb3CLUuA0I29DiqwhEvtCLWg0dA1KAA/tKuVHIFNfcozzGDFVeW/HoZvZvIcl38wUjSSqDN8FL/cfJZKccJNGKUwFajOU81roBR5Nej0UzVstfSd/tQa3sCq1R8vbh4Ci7kEiOKmRPpL3eF9sfT67igYtgGuzGI/ZjwNWPA0l875NrijAcmKW0qUitUTGEyzUr4LrtBXwVmYhZ5ocY+mXGMu5t5cJL4YrXbrHno67A8GkHUBRX5/OE64aI/SVq/DRdIXaa+TAhf9ZLsbC+ncl/pmHPQ/e/J+/++cHq9Ip5wIu2p2fbmAJKxfUD//6QE5FEsQUcfYjH0YKKaeDOo+TQoPbJMOG5nEbgyCn6z9DzBHG3tTwjhUPxPkuEyZ5WtLdJVphIMiaqBwQGhxx+zy/JpKrJLY/1xoa3nBtR/ehElRDGi13hcldUhNmM05Sl3rykq4qqRc863Cbkt6rYYDWSHJggj2gRUCEtMQR0pU7vUAuz0S201u5yGQHxeWftWXmzYll7b9aKoPtZRGPYPVj6tYG40hlOdQ+onJClOAL6+8cianZk6AgKWpiDwTQQI2ArNR9XHjeW1/o5e8yN/R61uRKl2GuJFn35kwGQCSivTQ6wB0oWZ+I7KCuLBYCvWaFVyWlo7BwcHPJz4cJmxRmqqg8EiM+RjXvdrL+HAtLHcI+2YTljWqe8zSGwRO4+DJX07ny+npxfQ0Grfr0vb/5HLOUyx8aCsKWa335/Og1U/5IGvh/5z3hmk6ao2gNxoM+h0+xgE+G/Nk0Bv0n5nPB5O5cziTbyG5l7vUp4NCsUDYChxy6hTfvux5kLHzHEduAsanbiqiZP14WW7eKhRVKR1q1Uasb1+1HDOsfRXmtAgty1Pv853lFY4mSZg0KLwTyAzOvFTofcDWAfCivi0puB34jqYrLhWn1xhYSGTIZTwXxyr8t7liPcts1MtJfZmjK4MlYUXANNGNtkDZ8O6KJHqz9g1AgdCmoHzIYOQK8O8w8DqnJbZGL8JxNFitgyovMInCYxpxSDKQYoDG4Fvs6ZEvfVEIuyh0JM2WnO6xadHhPU9ocbCjSktA/CnH6vkvqH8a+9rjnVc9xtHYZDfWyHefVD+9+OV/63TpqJAUAAA=', 'AUDIT-ADJOUMANI.json': 'H4sIAAAAAAAC/01RTW8TMRC991eMci7g3WSTDTmVUhASvUCFxGk1a48TI6+deuy2EeK/M96ED8kHa968efPe/LwCWHAsSdPiLSxuzI9YJgzuTU4Y2GN2MfBgEwV9GEaMxdPwrVlcV1qiJ8eCV6Iy2pJSetl2DW26nqjt+63ebhpcWtWMttdq1dkLM45M6YnMkOIzC71Vq+UMlOAeCw1HdOlcb7q5bsrRO435P6i/7IBmokF7dNO/eU2n1AwLiQILHoOVf66rTpTRYEa4//Swgy93N+/v7+DzuREmCrNluL2Fd99ffb2B1WsFGEztBheAcSJg0rXrYia5vZtD+OgjjwT0gtPRE+9AkjxBIi8NsTAckRn3xNdw5qCHEgwlf3JhD1I6ZIYQMxBnHL3jA5mzxmNB7/Jp+BNcVfswHwUOQvOVSgb4KEcDW1j+LuQIzzEZkcuphJqeAQlKUJ4dSdJcbYgUgczfh+qez4qyQS41ywW9aF+McG2KE+SDY5mHLtSdc3Lod8A5JmmIwZ8AWRwzYZLdvFxnNzs6loufimup1/6/Y+o5LrIHbLt1lbV2a4xuVCNPLXujqFu3Ztv12mqkkawAZttqs15149jYlRoVmqbbtNiv1Wa1uPr1Gy0KcfndAgAA', 'CREDITS.txt': 'H4sIAAAAAAAC/52Sy1LbQBBF9/6KXpIqe/QAmyqzkgW4DOYR7ARYNqOW1GGYcc2MDHxSviM/lrYIkEVW2WkeunP63lu6iuACbcVYuqcN6girfAXTP3tDKGSvpVGuUjU4dx2GwEM4dxYtnBHaUWmwE4m9PM0Pvgxhhq4zv37CDRmMVME1ejSGDJTOb5xHqJ0Xbd2yJVh7tEHusbMQMTyGKTzFWbkYZSodwuKsXC3WkE32xiJcOZ5mqRpneZbwDx04qt2TKpuM00zBMYoARShLKGL0/ND1qntb8mH30dmwIc01C9PDK2AXW+dF1nlu2KIBLbiNrCiAp4iCV6nB3LnGEKwurpaQwLxYL9ZXq6E8Mprdjw52kG2MmzBNkrZrRKepUZPSLqnecELS9ApJeHIGjuDUk9UtbBnhxDaGQwsb3rqoBrfFXbEU248Z9SPIIGhUb+pEINHf8XaaT1JJIT/cP1DwzXI/WXwFV8O8RYtv1oup6/8ivMUXNJfLa6F0dc2axZPoxQf5B8LGcIRILxGcNa9qcEM17WYhiew7hme0/A6dp9nhLgOyfQKLAIUxcO86uCSqjqBgyZzp83q+D/OvxRGsuLHOu7+V8jGc9DAiBoWX1kTSsfMS0qn0aOmeRzcUXOc1/atUarBuCWb8ILa8F8F/kEulwLoI2tkQOXaRQBuSvsphX9O6E/Dd0IlUnN2nHXLmqeLw0TM1+A3xgwz9SAMAAA=='}
ASSET_SHA256={'parallel-prepare.jsonl': 'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd', 'replies-prepare.jsonl': '29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1', 'PROVENANCE.json': '2551e624e4ca3674ad2ceabfb2b9c4621368b3fa475a9a578a14c845c4a28de6', 'RESSOURCES-BAOULE.json': '55dcee862196a54baddedea568f77f76083d33d7f670c0c25745983030ae5953', 'AUDIT-ADJOUMANI.json': '974b93d122712a7420d1d5cb36719fd3179fc0a177d5c80286665cf48a2badcf', 'CREDITS.txt': 'dabc90b83547dcc8eef10226be830c39b8886fe68067fa86cdaeade09686713d'}
DRIVE_DIRECTORY='/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape4'
from google.colab import drive
import pathlib,gzip,base64,hashlib
drive.mount('/content/drive')
OUT=pathlib.Path(DRIVE_DIRECTORY)
OUT.mkdir(parents=True,exist_ok=True)
for name,payload in PREPARED_ASSETS.items():
    data=gzip.decompress(base64.b64decode(payload))
    assert hashlib.sha256(data).hexdigest()==ASSET_SHA256[name]
    destination=OUT/name
    if destination.exists():
        assert hashlib.sha256(destination.read_bytes()).hexdigest()==ASSET_SHA256[name], 'Données existantes différentes : utiliser un nouveau dossier de run.'
    else:destination.write_bytes(data)
print('Données préparées vérifiées et sauvegardées dans',OUT)


## Lancer ou reprendre
Exécuter cette cellule avec un GPU Colab. Elle monte Drive, vérifie causalité/cache/reprise, extrait les colonnes texte WAXAL si nécessaire, puis entraîne les deux modèles. Chaque 100 étapes sauvegarde poids FP32, AdamW, RNG CPU/CUDA, sampler et compteurs. Elle refuse une reprise avec des empreintes de code, tokenizer ou données différentes.

Ne pas modifier les fichiers d'une expérience déjà commencée. Les chemins et empreintes du rapport permettent de retrouver le run exact.


In [ ]:
MODULE_SOURCES={'mandia_stage4.py': '"""Expérience S2S contrôlée : nouveaux poids, mêmes données/budgets pour 2 architectures.\nToutes les sauvegardes (poids, AdamW, RNG, compteur) vont directement dans Drive.\n"""\nimport json,pathlib,hashlib,random,time,math,gc,unicodedata,re,collections,contextlib\nfrom dataclasses import asdict\nimport torch\nfrom torch.nn import functional as F\nfrom mandia_seq2seq import S2SConfig,MandiaS2S,generate,state_bytes\nROOT=pathlib.Path(\'/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape4\')\nSTEPS,WARMUP,BATCH,LR=6000,200,16,3e-4\nSPECIAL=[\'<pad>\',\'<bos>\',\'<eos>\',\'<translate>\',\'<reply>\',\'<denoise>\',\'<mask>\']\nDEVICE=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\ndef sha(p):return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()\ndef canonical(s):return \' \'.join(unicodedata.normalize(\'NFC\',s).casefold().replace(\'’\',"\'").split())\ndef atomic_json(p,data):\n    p=pathlib.Path(p);tmp=p.with_suffix(\'.json.part\');tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2));tmp.replace(p)\ndef atomic_torch(p,data):\n    p=pathlib.Path(p);tmp=p.with_suffix(\'.pt.part\');torch.save(data,tmp);tmp.replace(p)\ndef context():return torch.autocast(\'cuda\',dtype=torch.bfloat16) if DEVICE.type==\'cuda\' else contextlib.nullcontext()\ndef sync():\n    if DEVICE.type==\'cuda\':torch.cuda.synchronize()\n\ndef prepare():\n    from tokenizers import Tokenizer,models,trainers,pre_tokenizers,decoders\n    rows=[json.loads(l) for l in (ROOT/\'parallel-prepare.jsonl\').read_text().splitlines()]\n    replies=[json.loads(l) for l in (ROOT/\'replies-prepare.jsonl\').read_text().splitlines()]\n    protected={canonical(r[k]) for r in rows if r[\'split\']!=\'train\' for k in [\'prompt\',\'answer\']}\n    for split in [\'train\',\'validation\',\'test\']:\n        for other in [\'train\',\'validation\',\'test\']:\n            if split>=other:continue\n            for field in [\'prompt\',\'answer\']:\n                a={canonical(r[field]) for r in rows if r[\'split\']==split}\n                b={canonical(r[field]) for r in rows if r[\'split\']==other}\n                assert not a&b,(split,other,field)\n    assert all(not any(canonical(r[k]) in protected for k in [\'prompt\',\'answer\']) for r in replies)\n    mono=[];waxal=ROOT/\'data/waxal-baoule-text.jsonl\'\n    if waxal.exists():\n        for line in waxal.read_text().splitlines():\n            r=json.loads(line)\n            for j,paragraph in enumerate(re.split(r\'\\n+|(?<=[.!?])\\s+\',r[\'text\'])):\n                t=\' \'.join(paragraph.split())\n                if 12<=len(t)<=900 and canonical(t) not in protected:\n                    mono.append({\'id\':r[\'id\']+f\'-{j}\',\'prompt\':t,\'answer\':t,\'task\':\'denoise\',\'domain\':\'monolingual\',\'split\':\'train\',\'source_document\':r[\'id\']})\n    assert mono,\'Extraire les transcriptions WAXAL avant cette expérience.\'\n    # Exclusion des longues références holdout apparaissant à l\'intérieur d\'une phrase.\n    hold=[p for p in protected if len(p)>=25]\n    mono=[r for r in mono if not any(p in canonical(r[\'answer\']) for p in hold)]\n    tokenpath=ROOT/\'tokenizer.json\'\n    if tokenpath.exists():tok=Tokenizer.from_file(str(tokenpath))\n    else:\n        tok=Tokenizer(models.BPE());tok.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False);tok.decoder=decoders.ByteLevel()\n        tr=trainers.BpeTrainer(vocab_size=4096,min_frequency=2,initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),special_tokens=SPECIAL)\n        texts=(r[k] for r in rows+replies+mono if r[\'split\']==\'train\' for k in [\'prompt\',\'answer\'])\n        tok.train_from_iterator(texts,trainer=tr);tok.save(str(tokenpath))\n    assert tok.decode(tok.encode(\'ɛ́ ɔ̀ ɲ á à ’ Français — Baoulé\').ids)==\'ɛ́ ɔ̀ ɲ á à ’ Français — Baoulé\'\n    groups={s:collections.defaultdict(list) for s in [\'train\',\'validation\',\'test\']};encoded=[];excluded=collections.Counter()\n    for r in rows+replies+mono:\n        src=[tok.token_to_id(\'<\'+r[\'task\']+\'>\')]+tok.encode(r[\'prompt\']).ids+[2]\n        target=tok.encode(r[\'answer\']).ids+[2]\n        if len(src)>160 or len(target)>192:\n            excluded[r[\'domain\']+\'/length\']+=1;continue\n        item={\'source\':src,\'target\':target,\'row\':r};groups[r[\'split\']][r[\'domain\']].append(item)\n        encoded.append({\'id\':r[\'id\'],\'split\':r[\'split\'],\'domain\':r[\'domain\'],\'source_tokens\':len(src),\'target_tokens\':len(target)})\n    for s in [\'train\',\'validation\',\'test\']:\n        for d in [\'everyday\',\'software\',\'gospel\',\'lexical\']:assert groups[s][d],(s,d)\n    assert groups[\'train\'][\'monolingual\'] and groups[\'train\'][\'reply\']\n    audit={\'rows\':{s:{d:len(items) for d,items in ds.items()} for s,ds in groups.items()},\'excluded\':dict(excluded),\'tokenizer_sha256\':sha(tokenpath),\'monolingual_source\':\'Google WAXAL official training split, text columns only\',\'no_holdout_tokenizer_training\':True,\'source_max\':160,\'target_max\':192,\'identity_denoising_is_not_translation\':True}\n    atomic_json(ROOT/\'DATA-AUDIT.json\',audit)\n    (ROOT/\'split-tokenise.jsonl\').write_text(\'\'.join(json.dumps(r)+\'\\n\' for r in encoded))\n    print(\'DATA_ETAPE4\',json.dumps(audit),flush=True)\n    return tok,groups\n\ndef batch(items,device,generator=None,denoise=False):\n    ss=[list(r[\'source\']) for r in items];tt=[r[\'target\'] for r in items]\n    if denoise:\n        for s in ss:\n            for i in range(1,len(s)-1):\n                if float(torch.rand((),generator=generator))<.25:s[i]=6\n    source=torch.zeros((len(ss),max(map(len,ss))),dtype=torch.long,device=device)\n    target=torch.zeros((len(tt),max(map(len,tt))),dtype=torch.long,device=device)\n    labels=torch.full_like(target,-100)\n    for i,(s,t) in enumerate(zip(ss,tt)):\n        source[i,:len(s)]=torch.tensor(s,device=device);target[i,:len(t)]=torch.tensor([1]+t[:-1],device=device);labels[i,:len(t)]=torch.tensor(t,device=device)\n    return source,target,labels,sum(map(len,ss))+sum(map(len,tt))\ndef choose(items,n,g):return [items[int(i)] for i in torch.randint(len(items),(n,),generator=g)]\ndef loss(model,source,target,labels,smoothing=0):\n    logits=model(source,target)\n    return F.cross_entropy(logits.float().reshape(-1,logits.size(-1)),labels.reshape(-1),ignore_index=-100,reduction=\'sum\',label_smoothing=smoothing),int((labels!=-100).sum())\n\n@torch.inference_mode()\ndef evaluate(model,groups,device):\n    model.eval();metrics={}\n    for domain in [\'everyday\',\'software\',\'gospel\',\'lexical\']:\n        total,count=0.,0\n        for start in range(0,len(groups[domain]),16):\n            s,t,y,_=batch(groups[domain][start:start+16],device)\n            with context():v,n=loss(model,s,t,y)\n            total+=float(v);count+=n\n        metrics[domain]={\'loss\':total/count,\'supervised_tokens\':count,\'rows\':len(groups[domain])}\n    return {\'selection_score\':sum(x[\'loss\'] for x in metrics.values())/4,\'domains\':metrics}\n\ndef train(model,groups,directory,signature,device,steps=STEPS):\n    directory.mkdir(parents=True,exist_ok=True)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,betas=(.9,.98),weight_decay=.01)\n    generator=torch.Generator().manual_seed(42)\n    last,best=directory/\'checkpoint-last.pt\',directory/\'checkpoint-best.pt\'\n    step,real,supervised,elapsed,history,bestscore=0,0,0,0.,[],float(\'inf\')\n    if last.exists():\n        saved=torch.load(last,map_location=\'cpu\',weights_only=True);assert saved[\'signature\']==signature\n        model.load_state_dict(saved[\'model\']);optimizer.load_state_dict(saved[\'optimizer\']);generator.set_state(saved[\'sampler_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if device.type==\'cuda\':torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        random.setstate(saved[\'python_rng\'])\n        step,real,supervised,elapsed,history,bestscore=(saved[k] for k in [\'step\',\'real_tokens\',\'supervised_tokens\',\'wall_seconds\',\'history\',\'best_score\'])\n        del saved;print(\'REPRISE_S2S\',directory.name,step,flush=True)\n    sync();started=time.perf_counter()\n    if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n    def save(done):\n        sync();payload={\'signature\':signature,\'config\':asdict(model.cfg),\'step\':done,\'target_steps\':steps,\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'sampler_rng\':generator.get_state(),\'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else [],\'python_rng\':random.getstate(),\'real_tokens\':real,\'supervised_tokens\':supervised,\'wall_seconds\':elapsed+time.perf_counter()-started,\'history\':history,\'best_score\':bestscore,\'precision\':\'bfloat16 autocast, FP32 weights/AdamW, no scaler needed\',\'scheduler\':\'200-step warmup then cosine to 10% LR; phase0 500 denoising steps\'}\n        atomic_torch(last,payload);atomic_json(directory/\'progress.json\',{k:payload[k] for k in [\'step\',\'target_steps\',\'real_tokens\',\'supervised_tokens\',\'wall_seconds\',\'best_score\',\'history\']})\n    if step==0:save(0)\n    for i in range(step,steps):\n        model.train();optimizer.zero_grad(set_to_none=True)\n        ratio=(i+1)/WARMUP if i<WARMUP else .1+.9*.5*(1+math.cos(math.pi*(i-WARMUP)/(steps-WARMUP)))\n        for paramgroup in optimizer.param_groups:paramgroup[\'lr\']=LR*ratio\n        if i<500:\n            specs=[(\'monolingual\',16,1.)]\n        else:\n            # Même séquence de tirages et de masques pour les deux variantes.\n            domains=[\'everyday\',\'software\',\'gospel\'];domain=domains[(i-500)%3]\n            specs=[(domain,16,.5),(\'lexical\',8,.2),(\'reply\',4,.1),(\'monolingual\',8,.2)]\n        objective=0.\n        for domain,n,weight in specs:\n            examples=choose(groups[\'train\'][domain],n,generator)\n            s,t,y,ntokens=batch(examples,device,generator,domain==\'monolingual\')\n            with context():value,tokens=loss(model,s,t,y,.05)\n            if not torch.isfinite(value):raise FloatingPointError(\'Perte non finie.\')\n            (weight*value/tokens).backward();objective+=weight*float(value.detach())/tokens;real+=ntokens;supervised+=tokens\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True);optimizer.step()\n        if (i+1)%100==0 or i==step:print(\'S2S_STEP\',directory.name,i+1,\'/\',steps,\'loss\',round(objective,4),flush=True)\n        if (i+1)%250==0 and i>=500 or i+1==steps:\n            ev=evaluate(model,groups[\'validation\'],device);score=ev[\'selection_score\']\n            history.append({\'step\':i+1,\'train_objective\':objective,\'validation\':ev,\'gradient_norm\':float(norm)})\n            print(\'S2S_VALIDATION\',directory.name,i+1,json.dumps(ev),flush=True)\n            if score<bestscore:\n                bestscore=score;atomic_torch(best,{\'signature\':signature,\'step\':i+1,\'config\':asdict(model.cfg),\'model\':model.state_dict(),\'score\':score})\n        if (i+1)%100==0 or i+1==steps:save(i+1)\n    sync();metrics={\'steps\':steps,\'real_tokens\':real,\'supervised_tokens\':supervised,\'wall_seconds_including_drive_io\':elapsed+time.perf_counter()-started,\'peak_gpu_bytes\':torch.cuda.max_memory_allocated() if device.type==\'cuda\' else None,\'best_validation_score\':bestscore,\'history\':history}\n    if step>=steps and (directory/\'train-metrics.json\').exists():metrics=json.loads((directory/\'train-metrics.json\').read_text())\n    selected=torch.load(best,map_location=\'cpu\',weights_only=True);model.load_state_dict(selected[\'model\']);metrics[\'best_step\']=selected[\'step\']\n    atomic_json(directory/\'train-metrics.json\',metrics);del optimizer,selected\n    model.zero_grad(set_to_none=True)\n    return metrics\n\n@torch.inference_mode()\ndef score_generation(model,items,tok,device,max_tokens=96):\n    from sacrebleu.metrics import CHRF\n    output=[]\n    for r in items:\n        with context():ids,_=generate(model,r[\'source\'],1,2,max_tokens)\n        output.append({\'id\':r[\'row\'][\'id\'],\'prompt_fr\':r[\'row\'][\'prompt\'],\'reference_bci\':r[\'row\'][\'answer\'],\'generated_bci\':tok.decode(ids),\'eos\':2 in ids,\'domain\':r[\'row\'][\'domain\'],\'native_validated\':False})\n    m=CHRF(word_order=2);score=m.corpus_score([r[\'generated_bci\'] for r in output],[[r[\'reference_bci\'] for r in output]])\n    return {\'rows\':len(output),\'chrF_pp\':score.score,\'exact_match\':sum(canonical(r[\'generated_bci\'])==canonical(r[\'reference_bci\']) for r in output)/len(output),\'no_eos\':sum(not r[\'eos\'] for r in output),\'metric_signature\':str(m.get_signature())},output\n\n@torch.inference_mode()\ndef benchmark(model,example,device):\n    model.eval();source=torch.tensor([example[\'source\']],device=device)\n    tokens=64;times=[];prefills=[]\n    for repeat in range(6):\n        sync();start=time.perf_counter()\n        with context():memory,valid=model.encode(source)\n        sync();prefill=time.perf_counter()-start;states=None;x=torch.tensor([[1]],device=device)\n        sync();start=time.perf_counter()\n        with context():\n            for offset in range(tokens):\n                logits,states=model.decode(x,memory,valid,states,offset,True);x=logits[:,-1].argmax(-1,keepdim=True)\n        sync();elapsed=time.perf_counter()-start\n        if repeat:times.append(elapsed);prefills.append(prefill)\n    times.sort();prefills.sort()\n    return {\'forced_tokens\':tokens,\'median_decode_seconds\':times[2],\'tokens_per_second\':tokens/times[2],\'median_encoder_seconds\':prefills[2],\'cache_bytes_after_64\':state_bytes(states),\'source_tokens\':source.shape[1],\'warmup\':1,\'repeats\':5,\'precision\':\'bf16\',\'cache_scope\':\'decoder self+cross K/V, excludes encoder hidden memory\'}\n\ndef main():\n    assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n    assert (ROOT/\'parallel-prepare.jsonl\').exists()\n    device=torch.device(\'cuda:0\');torch.set_num_threads(4)\n    tok,groups=prepare()\n    code=pathlib.Path(__file__).parent\n    fingerprint={\'code\':{p.name:sha(p) for p in code.glob(\'*.py\')},\'data\':{p.name:sha(p) for p in [ROOT/\'parallel-prepare.jsonl\',ROOT/\'replies-prepare.jsonl\',ROOT/\'tokenizer.json\',ROOT/\'data/waxal-baoule-text.jsonl\']},\'seed\':42,\'steps\':STEPS,\'selection\':\'unweighted mean of the four domain validation CE losses; no test-based selection\',\'initialization\':\'random new S2S weights; v0 checkpoint preserved; incompatible architecture, no weights transferred\'}\n    atomic_json(ROOT/\'EXPERIMENT.json\',fingerprint)\n    report={\'gpu\':torch.cuda.get_device_name(0),\'torch\':torch.__version__,\'seed\':42,\'models\':{},\'scope\':\'Research training. Direct/pivot translations plus three trained reply intents, not general dialogue; no native validation.\'}\n    for variant,ffn in [(\'hybrid\',768),(\'attention_reference\',728)]:\n        print(\'=== S2S\',variant,\'===\',flush=True)\n        torch.manual_seed(42);torch.cuda.manual_seed_all(42);random.seed(42)\n        cfg=S2SConfig(vocab_size=tok.get_vocab_size(),variant=\'hybrid\' if variant==\'hybrid\' else \'attention\',ffn_dim=ffn)\n        model=MandiaS2S(cfg).to(device);directory=ROOT/variant\n        desc=model.description();signature=hashlib.sha256(json.dumps({\'experiment\':fingerprint,\'config\':asdict(cfg)},sort_keys=True).encode()).hexdigest()\n        trained=train(model,groups,directory,signature,device)\n        test=evaluate(model,groups[\'test\'],device)\n        generations={};alloutputs=[]\n        for domain in [\'everyday\',\'software\',\'gospel\',\'lexical\']:\n            # Sélection fixe des 32 premiers IDs lexicalement triés, identique aux 2 modèles.\n            examples=sorted(groups[\'test\'][domain],key=lambda x:x[\'row\'][\'id\'])[:32]\n            metric,out=score_generation(model,examples,tok,device);generations[domain]=metric;alloutputs+=out\n        demos,demout=score_generation(model,groups[\'train\'][\'reply\'],tok,device,24)\n        probes=[]\n        for prompt,answer in [(\'Salut Mandia !\',\'Anouho\'),(\'Bonjour à toi\',\'Anouho\'),(\'Est-ce que tu vas bien ?\',\'mi wun ti kpa\'),(\'Tu te portes bien ?\',\'mi wun ti kpa\'),(\'Je dois partir, au revoir\',\'ye tɛ wo nu\'),(\'Au revoir et à bientôt\',\'ye tɛ wo nu\')]:\n            probes.append({\'source\':[4]+tok.encode(prompt).ids+[2],\'row\':{\'id\':\'probe-\'+str(len(probes)),\'prompt\':prompt,\'answer\':answer,\'domain\':\'reply_paraphrase_probe\'}})\n        probemetric,probeout=score_generation(model,probes,tok,device,24)\n        perf=benchmark(model,groups[\'test\'][\'everyday\'][0],device)\n        # Diagnostic de dépendance au français, même padding après permutation des sources.\n        items=groups[\'test\'][\'everyday\'][:32];s,t,y,_=batch(items,device)\n        with torch.inference_mode(),context():matched,n=loss(model,s,t,y);wrong,_=loss(model,s.roll(1,0),t,y)\n        dependence={\'examples\':len(items),\'matched_loss\':float(matched)/n,\'permuted_source_loss\':float(wrong)/n,\'loss_gap\':float(wrong-matched)/n,\'scope\':\'Diagnostic only, not native semantic correctness.\'}\n        metrics={\'description\':desc,\'training\':trained,\'test_all\':test,\'generation_test\':generations,\'training_reply_demos\':demos,\'reply_paraphrase_probes\':probemetric,\'performance\':perf,\'source_dependence\':dependence}\n        atomic_json(directory/\'metrics.json\',metrics);atomic_json(directory/\'test-generations.json\',alloutputs);atomic_json(directory/\'reply-demos.json\',{\'scope\':\'Seen training prompts, not an independent evaluation\',\'rows\':demout,\'paraphrase_probes\':probeout})\n        atomic_torch(directory/\'model.pt\',{\'config\':asdict(cfg),\'state_dict\':model.state_dict(),\'tokenizer_sha256\':sha(ROOT/\'tokenizer.json\')})\n        report[\'models\'][variant]=metrics;atomic_json(ROOT/\'rapport-etape4.json\',report)\n        print(\'S2S_RESULTAT\',variant,json.dumps({k:v for k,v in metrics.items() if k!=\'training\'}),flush=True)\n        print(\'S2S_REPLIES\',variant,json.dumps({\'training\':demout,\'paraphrase_probes\':probeout},ensure_ascii=False),flush=True)\n        del model;gc.collect();torch.cuda.empty_cache()\n    a,b=report[\'models\'].values();assert a[\'training\'][\'real_tokens\']==b[\'training\'][\'real_tokens\'];assert a[\'training\'][\'supervised_tokens\']==b[\'training\'][\'supervised_tokens\']\n    report[\'same_token_budget_verified\']=True\n    winner=min(report[\'models\'],key=lambda k:report[\'models\'][k][\'training\'][\'best_validation_score\'])\n    report[\'candidate_by_validation\']=winner\n    report[\'limitations\']=[\'one seed\',\'same tokens, not same GPU time\',\'no native speaker evaluation\',\'GATITOS French uses an English pivot\',\'limited trained reply intents\',\'Bible audio/text versions identified but not full imported; misleading HF bci-bible excluded\',\'published primitives, no novelty demonstrated\']\n    atomic_json(ROOT/\'rapport-etape4.json\',report)\n    files=[p for p in ROOT.rglob(\'*\') if p.is_file() and not any(x in p.parts for x in [\'__pycache__\']) and not p.name.endswith(\'.part\') and p.name!=\'SAUVEGARDE-VERIFIEE.json\']\n    receipt={\'root\':str(ROOT),\'files\':[{\'path\':str(p.relative_to(ROOT)),\'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in sorted(files)],\'verified_on_mounted_drive\':True,\'candidate_by_validation\':winner}\n    atomic_json(ROOT/\'SAUVEGARDE-VERIFIEE.json\',receipt)\n    print(\'MANDIA_S2S_REPORT_BEGIN\',json.dumps(report,ensure_ascii=False),\'MANDIA_S2S_REPORT_END\',flush=True)\n    print(\'MANDIA_S2S_SAVE_BEGIN\',json.dumps(receipt),\'MANDIA_S2S_SAVE_END\',flush=True)\n    return report\nif __name__==\'__main__\':main()\n', 'verify_resume_s2s.py': '"""Test de la vraie boucle : interruption après sauvegarde puis reprise 100+4."""\nimport pathlib,tempfile,torch,random,json\nimport mandia_stage4 as stage\nfrom mandia_seq2seq import MandiaS2S,S2SConfig\ndef main():\n    torch.set_num_threads(2);stage.DEVICE=torch.device(\'cpu\')\n    cfg=S2SConfig(vocab_size=32,dim=48,heads=6,kv_heads=2,ffn_dim=96,encoder_layers=1,decoder_layers=2,dropout=.1)\n    items=[{\'source\':[5,7+i,8+i,2],\'target\':[7+i,8+i,2],\'row\':{\'id\':str(i)}} for i in range(6)]\n    groups={\'train\':{\'monolingual\':items},\'validation\':{d:items for d in [\'everyday\',\'software\',\'gospel\',\'lexical\']}}\n    def model():torch.manual_seed(13);random.seed(13);return MandiaS2S(cfg)\n    with tempfile.TemporaryDirectory() as d:\n        a,b=pathlib.Path(d)/\'full\',pathlib.Path(d)/\'resume\'\n        stage.train(model(),groups,a,\'resume-test\',torch.device(\'cpu\'),steps=104)\n        original=stage.atomic_torch\n        class Interrupted(Exception):pass\n        def stop(path,payload):\n            original(path,payload)\n            if pathlib.Path(path).name==\'checkpoint-last.pt\' and payload.get(\'step\')==100:raise Interrupted()\n        stage.atomic_torch=stop\n        try:stage.train(model(),groups,b,\'resume-test\',torch.device(\'cpu\'),steps=104)\n        except Interrupted:pass\n        finally:stage.atomic_torch=original\n        stage.train(model(),groups,b,\'resume-test\',torch.device(\'cpu\'),steps=104)\n        x=torch.load(a/\'checkpoint-last.pt\',weights_only=True);y=torch.load(b/\'checkpoint-last.pt\',weights_only=True)\n        assert all(torch.equal(x[\'model\'][k],y[\'model\'][k]) for k in x[\'model\'])\n        assert torch.equal(x[\'sampler_rng\'],y[\'sampler_rng\']) and torch.equal(x[\'torch_rng\'],y[\'torch_rng\'])\n        assert x[\'real_tokens\']==y[\'real_tokens\'] and x[\'supervised_tokens\']==y[\'supervised_tokens\']\n        for k,v in x[\'optimizer\'][\'state\'].items():\n            assert all(torch.equal(t,y[\'optimizer\'][\'state\'][k][n]) if torch.is_tensor(t) else t==y[\'optimizer\'][\'state\'][k][n] for n,t in v.items())\n    result={\'cpu_actual_training_resume\':\'104 uninterrupted == 100+4 resumed, bitwise equal weights and AdamW\',\'sampler_and_dropout_rng\':\'equal\',\'token_counters\':\'equal\',\'gpu_scope\':\'CUDA RNG saved; GPU bitwise equivalence not asserted\'}\n    print(\'S2S_RESUME_TEST\',json.dumps(result),flush=True)\n    return result\nif __name__==\'__main__\':main()\n', 'mandia_seq2seq.py': '"""MandiaCompact S2S : implémentation Mandia, initialisation aléatoire.\nEncodeur bidirectionnel et accès à la source dans chaque bloc du décodeur.\nHybride : 4 convolutions causales + 2 GQA ; référence : 6 GQA.\nLes composants sont publiés ; cette combinaison n\'est pas une nouveauté prouvée.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom mandia_model import RMSNorm, Rotary, GatedCausalConv, SwiGLU\n\n@dataclass\nclass S2SConfig:\n    vocab_size: int = 4096\n    dim: int = 384\n    heads: int = 6\n    kv_heads: int = 2\n    encoder_layers: int = 2\n    decoder_layers: int = 6\n    ffn_dim: int = 768\n    variant: str = \'hybrid\'\n    conv_kernel: int = 5\n    context: int = 256\n    dropout: float = .1\n\nclass Attention(nn.Module):\n    def __init__(self,cfg):\n        super().__init__()\n        self.h,self.kvh,self.d=cfg.heads,cfg.kv_heads,cfg.dim//cfg.heads\n        self.q=nn.Linear(cfg.dim,cfg.dim,bias=False)\n        self.k=nn.Linear(cfg.dim,self.kvh*self.d,bias=False)\n        self.v=nn.Linear(cfg.dim,self.kvh*self.d,bias=False)\n        self.out=nn.Linear(cfg.dim,cfg.dim,bias=False)\n        self.rope=Rotary(self.d)\n    def keys(self,x):\n        b,n,_=x.shape\n        return tuple(layer(x).view(b,n,self.kvh,self.d).transpose(1,2) for layer in (self.k,self.v))\n    def forward(self,x,source=None,valid=None,state=None,offset=0,cache=False,causal=False):\n        b,n,_=x.shape\n        q=self.q(x).view(b,n,self.h,self.d).transpose(1,2)\n        past=0\n        if source is not None:\n            k,v=state if state is not None else self.keys(source)\n        else:\n            k,v=self.keys(x);q,k=self.rope(q,offset),self.rope(k,offset)\n            if state is not None:\n                past=state[0].shape[-2];k,v=torch.cat((state[0],k),-2),torch.cat((state[1],v),-2)\n        mask=None\n        if valid is not None:mask=valid[:,None,None,:]\n        use_causal=causal and past==0\n        if causal and past and n>1:\n            mask=torch.arange(k.size(-2),device=x.device)[None,:] <= torch.arange(offset,offset+n,device=x.device)[:,None]\n        y=F.scaled_dot_product_attention(q,k.repeat_interleave(self.h//self.kvh,1),v.repeat_interleave(self.h//self.kvh,1),attn_mask=mask,is_causal=use_causal)\n        return self.out(y.transpose(1,2).contiguous().view(b,n,-1)),((k,v) if cache else None)\n\nclass EncoderBlock(nn.Module):\n    def __init__(self,cfg):\n        super().__init__();self.norm1=RMSNorm(cfg.dim);self.norm2=RMSNorm(cfg.dim)\n        self.attn=Attention(cfg);self.ffn=SwiGLU(cfg);self.drop=nn.Dropout(cfg.dropout)\n    def forward(self,x,valid):\n        y,_=self.attn(self.norm1(x),valid=valid);x=x+self.drop(y)\n        return x+self.drop(self.ffn(self.norm2(x)))\n\nclass DecoderBlock(nn.Module):\n    def __init__(self,cfg,index):\n        super().__init__();self.kind=\'conv\' if cfg.variant==\'hybrid\' and index%3!=2 else \'gqa\'\n        self.norm1,self.norm2,self.norm3=RMSNorm(cfg.dim),RMSNorm(cfg.dim),RMSNorm(cfg.dim)\n        self.mix=GatedCausalConv(cfg) if self.kind==\'conv\' else Attention(cfg)\n        self.cross=Attention(cfg);self.ffn=SwiGLU(cfg);self.drop=nn.Dropout(cfg.dropout)\n    def forward(self,x,source,valid,state=None,offset=0,cache=False):\n        selfstate,crossstate=(None,None) if state is None else state\n        if self.kind==\'conv\':y,selfstate=self.mix(self.norm1(x),selfstate,offset,cache)\n        else:y,selfstate=self.mix(self.norm1(x),state=selfstate,offset=offset,cache=cache,causal=True)\n        x=x+self.drop(y)\n        y,crossstate=self.cross(self.norm2(x),source=source,valid=valid,state=crossstate,cache=cache)\n        x=x+self.drop(y);x=x+self.drop(self.ffn(self.norm3(x)))\n        return x,(selfstate,crossstate)\n\nclass MandiaS2S(nn.Module):\n    def __init__(self,cfg):\n        super().__init__();self.cfg=cfg\n        self.embedding=nn.Embedding(cfg.vocab_size,cfg.dim)\n        self.encoder=nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.encoder_layers)])\n        self.decoder=nn.ModuleList([DecoderBlock(cfg,i) for i in range(cfg.decoder_layers)])\n        self.encnorm,self.decnorm=RMSNorm(cfg.dim),RMSNorm(cfg.dim)\n        self.apply(self._init)\n        for name,p in self.named_parameters():\n            if name.endswith((\'out.weight\',\'down.weight\')):nn.init.normal_(p,std=.02/math.sqrt(2*(cfg.encoder_layers+cfg.decoder_layers)))\n    @staticmethod\n    def _init(m):\n        if isinstance(m,(nn.Linear,nn.Embedding,nn.Conv1d)):nn.init.normal_(m.weight,std=.02)\n    def encode(self,source,pad=0):\n        valid=source!=pad\n        if not valid.any(-1).all():raise ValueError(\'Source vide.\')\n        x=self.embedding(source)\n        for block in self.encoder:x=block(x,valid)\n        return self.encnorm(x),valid\n    def decode(self,target,source,valid,states=None,offset=0,cache=False):\n        if offset+target.shape[1]>self.cfg.context:raise ValueError(\'Contexte décodeur dépassé.\')\n        x=self.embedding(target);out=[]\n        for i,block in enumerate(self.decoder):\n            x,s=block(x,source,valid,None if states is None else states[i],offset,cache);out.append(s)\n        logits=F.linear(self.decnorm(x),self.embedding.weight)\n        return (logits,out) if cache else logits\n    def forward(self,source,target):\n        memory,valid=self.encode(source);return self.decode(target,memory,valid)\n    def description(self):\n        return {\'config\':asdict(self.cfg),\'parameters\':sum(p.numel() for p in self.parameters()),\'decoder_mix\':[b.kind for b in self.decoder],\'initialization\':\'random, no pretrained model\'}\n\n@torch.inference_mode()\ndef generate(model,source,bos,eos,max_tokens=96):\n    model.eval();dev=next(model.parameters()).device\n    if isinstance(source,list):source=torch.tensor([source],device=dev)\n    memory,valid=model.encode(source);states=None;token=torch.tensor([[bos]],device=dev);answer=[]\n    for offset in range(max_tokens):\n        logits,states=model.decode(token,memory,valid,states,offset,True)\n        value=int(logits[0,-1].float().argmax());answer.append(value)\n        if value==eos:break\n        token=torch.tensor([[value]],device=dev)\n    return answer,states\n\ndef state_bytes(states):\n    if states is None:return 0\n    if torch.is_tensor(states):return states.numel()*states.element_size()\n    return sum(state_bytes(x) for x in states)\n', 'waxal_text.py': '"""Extraction de colonnes texte WAXAL, sans télécharger les octets audio.\nLe manifeste garde les références des shards pour une future expérience vocale.\n"""\nimport io,json,pathlib,requests,hashlib,time\nimport pyarrow.parquet as pq\nREVISION=\'5f4d8ca24f2b9d168b2ee545f1febaaff4b40580\'\n\nclass RemoteRanges(io.RawIOBase):\n    def __init__(self,url):\n        self.url=url;self.pos=0;self.transferred=0;self.session=requests.Session()\n        r=self.session.get(url,headers={\'Range\':\'bytes=0-3\'},timeout=90)\n        if r.status_code!=206:raise RuntimeError(\'Le serveur ne permet pas une lecture partielle ; téléchargement audio refusé.\')\n        self.size=int(r.headers[\'Content-Range\'].split(\'/\')[-1]);self.transferred+=len(r.content)\n    def readable(self):return True\n    def seekable(self):return True\n    def tell(self):return self.pos\n    def seek(self,offset,whence=0):\n        self.pos=offset if whence==0 else self.pos+offset if whence==1 else self.size+offset\n        return self.pos\n    def read(self,n=-1):\n        if n<0:n=self.size-self.pos\n        n=min(n,self.size-self.pos)\n        if n<=0:return b\'\'\n        for retry in range(4):\n            r=self.session.get(self.url,headers={\'Range\':f\'bytes={self.pos}-{self.pos+n-1}\'},timeout=90)\n            if r.status_code==206:break\n            if retry==3:r.raise_for_status();raise RuntimeError(\'Range ignoré.\')\n            time.sleep(1+retry)\n        data=r.content\n        if len(data)!=n:raise RuntimeError(\'Lecture partielle incorrecte.\')\n        self.transferred+=n;self.pos+=n;return data\n    def readinto(self,b):\n        data=self.read(len(b));b[:len(data)]=data;return len(data)\n\ndef extract(directory):\n    directory=pathlib.Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    destination=directory/\'waxal-baoule-text.jsonl\';receipt=directory/\'WAXAL-TEXTE.json\'\n    if destination.exists() and receipt.exists():\n        old=json.loads(receipt.read_text())\n        if hashlib.sha256(destination.read_bytes()).hexdigest()==old[\'sha256\']:return destination\n    meta=requests.get(f\'https://huggingface.co/api/datasets/google/WaxalNLP/revision/{REVISION}\',timeout=60)\n    meta.raise_for_status();meta=meta.json();assert meta[\'sha\']==REVISION\n    files=sorted(x[\'rfilename\'] for x in meta[\'siblings\'] if x[\'rfilename\'].startswith(\'data/TTS/bau/bau-train-\') and x[\'rfilename\'].endswith(\'.parquet\'))\n    assert files\n    rows=[];shards=[]\n    for filename in files:\n        url=f\'https://huggingface.co/datasets/google/WaxalNLP/resolve/{REVISION}/{filename}\'\n        remote=RemoteRanges(url)\n        table=pq.read_table(remote,columns=[\'id\',\'text\',\'speaker_id\',\'locale\'],pre_buffer=False)\n        for row in table.to_pylist():\n            assert row[\'locale\']==\'bau\',row\n            rows.append(dict(row,source_file=filename,source_dataset=\'google/WaxalNLP\',revision=REVISION,original_split=\'train\',license=\'CC-BY-4.0\',audio_downloaded=False))\n        shards.append({\'file\':filename,\'rows\':len(table),\'remote_bytes\':remote.size,\'transferred_bytes\':remote.transferred})\n        print(\'WAXAL_TEXTE\',filename,len(table),remote.transferred,flush=True)\n    part=destination.with_suffix(\'.part\');part.write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows));part.replace(destination)\n    receipt.write_text(json.dumps({\'source\':\'https://huggingface.co/datasets/google/WaxalNLP\',\'revision\':REVISION,\'provider\':\'University of Ghana, Google WAXAL\',\'config\':\'bau_tts (Baoule in publisher card)\',\'license\':\'CC-BY-4.0\',\'rows\':len(rows),\'sha256\':hashlib.sha256(destination.read_bytes()).hexdigest(),\'shards\':shards,\'audio_downloaded\':False,\'usage\':\'Baoulé denoising training; not French/BCI parallel examples\',\'note\':\'Only official training split; text exact overlaps with all parallel holdouts excluded before training.\'},ensure_ascii=False,indent=2))\n    return destination\n', 'verify_seq2seq.py': "import torch,tempfile,pathlib\nfrom mandia_seq2seq import MandiaS2S,S2SConfig\n\ndef main():\n    torch.set_num_threads(2)\n    results=[]\n    for variant in ['hybrid','attention']:\n        torch.manual_seed(9)\n        m=MandiaS2S(S2SConfig(vocab_size=32,dim=48,heads=6,kv_heads=2,ffn_dim=96,encoder_layers=2,decoder_layers=3,variant=variant,dropout=0)).eval()\n        s=torch.tensor([[1,3,5,7],[1,4,6,0]]);t=torch.tensor([[1,9,10,11],[1,12,13,0]])\n        full=m(s,t)\n        future=t.clone();future[:,3]=20\n        torch.testing.assert_close(full[:,:3],m(s,future)[:,:3],atol=1e-5,rtol=1e-5)\n        extended=torch.cat([s,torch.zeros((2,3),dtype=torch.long)],1)\n        torch.testing.assert_close(full,m(extended,t),atol=1e-5,rtol=1e-5)\n        memory,valid=m.encode(s);states=None;values=[]\n        for i in range(t.shape[1]):\n            logits,states=m.decode(t[:,i:i+1],memory,valid,states,i,True);values.append(logits)\n        torch.testing.assert_close(full,torch.cat(values,1),atol=1e-5,rtol=1e-5)\n        first,states=m.decode(t[:,:2],memory,valid,cache=True)\n        rest,_=m.decode(t[:,2:],memory,valid,states,2,True)\n        torch.testing.assert_close(full,torch.cat([first,rest],1),atol=1e-5,rtol=1e-5)\n        changed=s.clone();changed[:,1]=21\n        assert (full-m(changed,t)).abs().max()>1e-5\n        m.train();loss=m(s,t).square().mean();loss.backward()\n        assert all(p.grad is None or torch.isfinite(p.grad).all() for p in m.parameters())\n        assert m.encoder[0].attn.q.weight.grad.abs().sum()>0\n        results.append({'variant':variant,'causality':True,'padding_invariance':True,'token_and_chunk_cache':True,'source_effect':True,'encoder_gradients':True})\n    print('S2S_VERIFICATION',results,flush=True)\n    return results\nif __name__=='__main__':main()\n", 'mandia_model.py': '"""MandiaCompact v0 : prototype causal écrit pour Mandia, poids initialisés aléatoirement.\nBriques connues : RMSNorm, RoPE, SwiGLU, GQA et convolution causale courte.\nLa combinaison proposée est une hypothèse expérimentale, pas une nouveauté démontrée.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n@dataclass\nclass ModelConfig:\n    vocab_size: int = 4096\n    dim: int = 384\n    layers: int = 8\n    heads: int = 6\n    kv_heads: int = 2\n    ffn_dim: int = 768\n    context: int = 512\n    conv_kernel: int = 5\n    attention_every: int = 4\n    variant: str = "hybrid"\n    tie_embeddings: bool = True\n    def __post_init__(self):\n        assert self.dim % self.heads == 0\n        assert self.heads % self.kv_heads == 0\n        assert (self.dim // self.heads) % 2 == 0\n        assert self.variant in {"hybrid", "attention", "conv_only"}\n        assert self.conv_kernel >= 1 and self.attention_every >= 1\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(dim))\n    def forward(self, x):\n        scale = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6)\n        return (x.float() * scale).to(x.dtype) * self.weight.to(x.dtype)\n\nclass Rotary(nn.Module):\n    def __init__(self, dim):\n        super().__init__()\n        self.register_buffer("inv_freq", 1.0 / (10000 ** (torch.arange(0, dim, 2).float()/dim)), persistent=False)\n    def forward(self, x, offset):\n        positions = torch.arange(offset, offset+x.size(-2), device=x.device, dtype=torch.float32)\n        angle = positions[:, None] * self.inv_freq[None, :]\n        cos, sin = angle.cos().to(x.dtype), angle.sin().to(x.dtype)\n        a, b = x[..., 0::2], x[..., 1::2]\n        return torch.stack((a*cos-b*sin, a*sin+b*cos), dim=-1).flatten(-2)\n\nclass GroupedAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.heads, self.kv_heads, self.head_dim = cfg.heads, cfg.kv_heads, cfg.dim//cfg.heads\n        self.q = nn.Linear(cfg.dim, cfg.dim, bias=False)\n        self.k = nn.Linear(cfg.dim, cfg.kv_heads*self.head_dim, bias=False)\n        self.v = nn.Linear(cfg.dim, cfg.kv_heads*self.head_dim, bias=False)\n        self.out = nn.Linear(cfg.dim, cfg.dim, bias=False)\n        self.rope = Rotary(self.head_dim)\n    def forward(self, x, state=None, offset=0, cache=False):\n        batch, length, _ = x.shape\n        q = self.q(x).view(batch,length,self.heads,self.head_dim).transpose(1,2)\n        k = self.k(x).view(batch,length,self.kv_heads,self.head_dim).transpose(1,2)\n        v = self.v(x).view(batch,length,self.kv_heads,self.head_dim).transpose(1,2)\n        q, k = self.rope(q, offset), self.rope(k, offset)\n        past = 0\n        if state is not None:\n            past = state[0].size(-2)\n            k, v = torch.cat((state[0],k),dim=-2), torch.cat((state[1],v),dim=-2)\n        new_state = (k, v) if cache else None\n        # Le cache reste en kv_heads ; cette implémentation portable répète ensuite K/V pour SDPA.\n        kr = k.repeat_interleave(self.heads//self.kv_heads,dim=1)\n        vr = v.repeat_interleave(self.heads//self.kv_heads,dim=1)\n        mask, causal = None, past == 0\n        if past and length > 1:\n            qi = torch.arange(length,device=x.device)[:,None] + past\n            ki = torch.arange(k.size(-2),device=x.device)[None,:]\n            mask = ki <= qi\n        y = F.scaled_dot_product_attention(q,kr,vr,attn_mask=mask,dropout_p=0.0,is_causal=causal)\n        y = y.transpose(1,2).contiguous().view(batch,length,-1)\n        return self.out(y), new_state\n\nclass GatedCausalConv(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.kernel = cfg.conv_kernel\n        self.proj = nn.Linear(cfg.dim,cfg.dim,bias=False)\n        self.conv = nn.Conv1d(cfg.dim,cfg.dim,cfg.conv_kernel,groups=cfg.dim,bias=False)\n        self.gate_weight = nn.Parameter(torch.zeros(cfg.dim))\n        self.gate_bias = nn.Parameter(torch.zeros(cfg.dim))\n        self.out = nn.Linear(cfg.dim,cfg.dim,bias=False)\n    def forward(self, x, state=None, offset=0, cache=False):\n        u = self.proj(x).transpose(1,2)\n        if state is None:\n            state = u.new_zeros(u.size(0),u.size(1),self.kernel-1)\n        sequence = torch.cat((state,u),dim=-1)\n        y = self.conv(sequence).transpose(1,2)\n        gate = torch.sigmoid(x*self.gate_weight.to(x.dtype)+self.gate_bias.to(x.dtype))\n        y = F.silu(y)*gate\n        new_state = sequence[:,:,-(self.kernel-1):] if self.kernel > 1 else sequence[:,:,:0]\n        return self.out(y), new_state.clone() if cache else None\n\nclass SwiGLU(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.gate = nn.Linear(cfg.dim,cfg.ffn_dim,bias=False)\n        self.up = nn.Linear(cfg.dim,cfg.ffn_dim,bias=False)\n        self.down = nn.Linear(cfg.ffn_dim,cfg.dim,bias=False)\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x))*self.up(x))\n\nclass Block(nn.Module):\n    def __init__(self, cfg, index):\n        super().__init__()\n        attention = cfg.variant == "attention" or (cfg.variant == "hybrid" and (index+1)%cfg.attention_every == 0)\n        self.kind = "gqa" if attention else "conv"\n        self.norm1, self.norm2 = RMSNorm(cfg.dim), RMSNorm(cfg.dim)\n        self.mix = GroupedAttention(cfg) if attention else GatedCausalConv(cfg)\n        self.ffn = SwiGLU(cfg)\n    def forward(self, x, state=None, offset=0, cache=False):\n        delta, state = self.mix(self.norm1(x),state,offset,cache)\n        x = x+delta\n        x = x+self.ffn(self.norm2(x))\n        return x, state\n\nclass MandiaLM(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.embedding = nn.Embedding(cfg.vocab_size,cfg.dim)\n        self.blocks = nn.ModuleList([Block(cfg,i) for i in range(cfg.layers)])\n        self.norm = RMSNorm(cfg.dim)\n        self.head = None if cfg.tie_embeddings else nn.Linear(cfg.dim,cfg.vocab_size,bias=False)\n        self.apply(self._init)\n        for name, parameter in self.named_parameters():\n            if name.endswith(("mix.out.weight", "ffn.down.weight")):\n                nn.init.normal_(parameter,std=0.02/math.sqrt(2*cfg.layers))\n    @staticmethod\n    def _init(module):\n        if isinstance(module,(nn.Linear,nn.Embedding,nn.Conv1d)):\n            nn.init.normal_(module.weight,std=0.02)\n    def forward(self, input_ids, states=None, offset=0, cache=False):\n        if offset+input_ids.size(1) > self.cfg.context:\n            raise ValueError("Contexte dépassé : raccourcir la demande ou la génération.")\n        x, new_states = self.embedding(input_ids), []\n        for i, block in enumerate(self.blocks):\n            x, state = block(x,None if states is None else states[i],offset,cache)\n            new_states.append(state)\n        x = self.norm(x)\n        logits = F.linear(x,self.embedding.weight) if self.head is None else self.head(x)\n        return (logits,new_states) if cache else logits\n    def parameter_count(self):\n        return sum(p.numel() for p in self.parameters())\n    def description(self):\n        return {"config":asdict(self.cfg),"parameters":self.parameter_count(),\n                "blocks":[b.kind for b in self.blocks],"initialization":"random, no pretrained weights"}\n\n@torch.inference_mode()\ndef generate_ids(model, prefix, eos_id, max_new_tokens=64, temperature=0.0):\n    model.eval()\n    device = next(model.parameters()).device\n    x = torch.tensor([prefix],dtype=torch.long,device=device)\n    logits, states = model(x,cache=True)\n    generated, offset = [], len(prefix)\n    for i in range(min(max_new_tokens,model.cfg.context-len(prefix))):\n        last = logits[0,-1].float()\n        token = int(last.argmax()) if temperature <= 0 else int(torch.multinomial(F.softmax(last/temperature,dim=-1),1))\n        generated.append(token)\n        if token == eos_id:\n            break\n        if i+1 < min(max_new_tokens,model.cfg.context-len(prefix)):\n            logits, states = model(torch.tensor([[token]],device=device),states,offset,cache=True)\n            offset += 1\n    return generated\n\ndef cache_bytes(states):\n    return sum(t.numel()*t.element_size() for state in states for t in (state if isinstance(state,tuple) else [state]) if t is not None)\n'}
DRIVE_DIRECTORY='/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape4'
import pathlib,sys,json,hashlib,torch
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Utiliser un GPU compatible bf16.'
OUT=pathlib.Path(DRIVE_DIRECTORY)
assert OUT.exists() and (OUT/'parallel-prepare.jsonl').exists(), 'Les données préparées doivent être dans le dossier Drive de cette expérience.'
CODE=OUT/'code';CODE.mkdir(exist_ok=True)
for name,source in MODULE_SOURCES.items():
    (CODE/name).write_text(source)
sys.path.insert(0,str(CODE))
for name in ['mandia_model','mandia_seq2seq','mandia_stage4','waxal_text','verify_seq2seq','verify_resume_s2s']:
    sys.modules.pop(name,None)
from verify_seq2seq import main as check_architecture
from verify_resume_s2s import main as check_resume
checks=check_architecture()
resume=check_resume()
(OUT/'VERIFICATION-CODE.json').write_text(json.dumps({'architecture':checks,'resume':resume},ensure_ascii=False,indent=2))
from waxal_text import extract
extract(OUT/'data')
import mandia_stage4
mandia_stage4.DEVICE=torch.device('cuda:0')
report=mandia_stage4.main()


## Tester les poids sélectionnés
La sélection utilise la validation, puis le test est consulté une fois. Les exemples « Bonjour » et « Comment vas-tu ? » sont des démonstrations d'entraînement, pas des preuves de généralisation. Le mode `reply` n'a appris que trois intentions ; le mode `translate` traduit un texte et ne répond pas à une question.


In [ ]:
import torch,json
from tokenizers import Tokenizer
import sys
sys.path.insert(0,str(OUT/"code"))
from mandia_seq2seq import MandiaS2S,S2SConfig,generate
report=json.loads((OUT/'rapport-etape4.json').read_text())
candidate=report.get('candidate_by_validation','hybrid')
if 'candidate_by_validation' not in report:
    print('Comparaison encore incomplète : démonstration du modèle hybride déjà sauvegardé.')
payload=torch.load(OUT/candidate/'model.pt',map_location='cpu',weights_only=True)
model=MandiaS2S(S2SConfig(**payload['config'])).to('cuda').eval()
model.load_state_dict(payload['state_dict'])
tokenizer=Tokenizer.from_file(str(OUT/'tokenizer.json'))
import hashlib
assert hashlib.sha256((OUT/'tokenizer.json').read_bytes()).hexdigest()==payload['tokenizer_sha256'], 'Tokenizer incompatible avec les poids.'
assert [tokenizer.token_to_id(t) for t in ['<pad>','<bos>','<eos>','<translate>','<reply>','<denoise>','<mask>']]==list(range(7))
def repondre_francais(texte,mode='reply'):
    assert mode in ['reply','translate']
    source=[tokenizer.token_to_id('<'+mode+'>')]+tokenizer.encode(texte).ids+[2]
    if len(source)>160:raise ValueError('Texte trop long pour ce prototype.')
    with torch.autocast('cuda',dtype=torch.bfloat16):tokens,_=generate(model,source,1,2,96)
    return tokenizer.decode(tokens)
for phrase in ['Bonjour !','Comment vas-tu ?','Au revoir']:
    print(phrase,'→',repondre_francais(phrase))
print('Traduction :',repondre_francais('merci',mode='translate'))


## Export CPU int8 vérifié
L’export a été exécuté dans Colab. Accepté par validation : True. Détails dans le bilan et `QUANTIFICATION-VERIFIEE.json`. Les checkpoints FP32 et AdamW restent conservés.


In [ ]:
import pathlib,sys
root=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape4')
exports=root/"export-cpu"
exports.mkdir(exist_ok=True)
(exports/"quantifier_cpu.py").write_text('"""Export CPU dynamique int8 vérifié sur validation, pas choisi sur le test.\nPoids source FP32 et checkpoints de reprise conservés dans Drive.\nUtilise le backend quantifié PyTorch disponible ; API actuelle dépréciée.\n"""\nimport json,pathlib,hashlib,time,statistics,copy,torch,warnings,platform\nfrom dataclasses import asdict\nfrom tokenizers import Tokenizer\nfrom mandia_seq2seq import MandiaS2S,S2SConfig,generate\nimport mandia_stage4 as stage\n\n@torch.inference_mode()\ndef replies(model,items,tok):\n    output=[]\n    for r in items:\n        ids,_=generate(model,r[\'source\'],1,2,24)\n        output.append({\'id\':r[\'row\'][\'id\'],\'prompt_fr\':r[\'row\'][\'prompt\'],\'reference_bci\':r[\'row\'][\'answer\'],\'generated_bci\':tok.decode(ids),\'eos\':2 in ids})\n    return output\n\n@torch.inference_mode()\ndef cpu_benchmark(model,source):\n    source=torch.tensor([source]);times=[]\n    for repeat in range(4):\n        memory,valid=model.encode(source);states=None;x=torch.tensor([[1]]);start=time.perf_counter()\n        for offset in range(64):\n            logits,states=model.decode(x,memory,valid,states,offset,True);x=logits[:,-1].argmax(-1,keepdim=True)\n        if repeat:times.append(time.perf_counter()-start)\n    median=statistics.median(times)\n    return {\'forced_decode_tokens\':64,\'threads\':torch.get_num_threads(),\'warmup\':1,\'repeats\':3,\'median_decode_seconds\':median,\'tokens_per_second\':64/median,\'device\':\'Colab CPU, not the user Mac\',\'precision\':\'FP32 activations, dynamic int8 Linear weights if quantized\'}\n\ndef main():\n    root=stage.ROOT;folder=root/\'export-cpu\';folder.mkdir(exist_ok=True)\n    report=json.loads((root/\'rapport-etape4.json\').read_text());candidate=report[\'candidate_by_validation\']\n    source=root/candidate/\'model.pt\';payload=torch.load(source,map_location=\'cpu\',weights_only=True)\n    tok=Tokenizer.from_file(str(root/\'tokenizer.json\'))\n    assert stage.sha(root/\'tokenizer.json\')==payload[\'tokenizer_sha256\']\n    old_device=stage.DEVICE;stage.DEVICE=torch.device(\'cpu\');threads=torch.get_num_threads();torch.set_num_threads(4)\n    try:\n        tok,groups=stage.prepare()\n        model=MandiaS2S(S2SConfig(**payload[\'config\'])).eval();model.load_state_dict(payload[\'state_dict\'])\n        f32valid=stage.evaluate(model,groups[\'validation\'],torch.device(\'cpu\'))\n        f32reply=replies(model,groups[\'train\'][\'reply\'],tok)\n        f32speed=cpu_benchmark(model,groups[\'test\'][\'everyday\'][0][\'source\'])\n        engines=[x for x in [\'x86\',\'fbgemm\',\'qnnpack\'] if x in torch.backends.quantized.supported_engines]\n        assert engines,\'Aucun backend int8 disponible.\'\n        torch.backends.quantized.engine=engines[0]\n        with warnings.catch_warnings(record=True) as recorded:\n            warnings.simplefilter(\'always\')\n            quant=torch.ao.quantization.quantize_dynamic(model,{torch.nn.Linear},dtype=torch.qint8,inplace=False).eval()\n        qvalid=stage.evaluate(quant,groups[\'validation\'],torch.device(\'cpu\'))\n        qreply=replies(quant,groups[\'train\'][\'reply\'],tok)\n        qspeed=cpu_benchmark(quant,groups[\'test\'][\'everyday\'][0][\'source\'])\n        delta=qvalid[\'selection_score\']-f32valid[\'selection_score\']\n        unchanged=all(stage.canonical(a[\'generated_bci\'])==stage.canonical(b[\'generated_bci\']) for a,b in zip(f32reply,qreply))\n        correct=all(stage.canonical(a[\'generated_bci\'])==stage.canonical(a[\'reference_bci\']) for a in qreply)\n        accepted=delta<=.05 and unchanged and correct\n        destination=folder/\'model-dynamic-int8-cpu.pt\'\n        stage.atomic_torch(destination,{\'config\':payload[\'config\'],\'state_dict\':quant.state_dict(),\'tokenizer_sha256\':payload[\'tokenizer_sha256\'],\'format\':\'PyTorch eager dynamic int8 Linear state_dict; embedding/Conv1d/norm remain FP32\',\'engine_tested\':engines[0],\'torch_version\':str(torch.__version__),\'source_model_sha256\':stage.sha(source),\'accepted_by_validation\':accepted})\n        # Relire le vrai export et reconstruire les couches quantifiées.\n        saved=torch.load(destination,map_location=\'cpu\',weights_only=True)\n        with warnings.catch_warnings():\n            warnings.simplefilter(\'ignore\',DeprecationWarning)\n            restored=torch.ao.quantization.quantize_dynamic(MandiaS2S(S2SConfig(**saved[\'config\'])).eval(),{torch.nn.Linear},dtype=torch.qint8)\n        restored.load_state_dict(saved[\'state_dict\'])\n        assert replies(restored,groups[\'train\'][\'reply\'][:3],tok)==qreply[:3]\n        result={\'candidate\':candidate,\'accepted\':accepted,\'selection\':\'validation CE delta <=0.05 and all 14 trained reply outputs unchanged/correct; no test-based selection\',\'validation_fp32\':f32valid,\'validation_int8\':qvalid,\'validation_delta\':delta,\'trained_replies_unchanged\':unchanged,\'trained_replies_correct\':correct,\'reply_count\':len(qreply),\'reply_outputs\':qreply,\'cpu_fp32\':f32speed,\'cpu_dynamic_int8\':qspeed,\'source_fp32_file_bytes\':source.stat().st_size,\'int8_file_bytes\':destination.stat().st_size,\'int8_file_sha256\':stage.sha(destination),\'size_fraction\':destination.stat().st_size/source.stat().st_size,\'engine\':engines[0],\'torch\':torch.__version__,\'safe_state_dict_reload_checked\':True,\'scope\':\'Experimental Colab CPU export; no guarantee of speed or backend support on other hardware; API deprecated, migration needed for future torch versions. Parameter count unchanged; serialization size reduced.\'}\n        stage.atomic_json(folder/\'QUANTIFICATION-VERIFIEE.json\',result)\n        print(\'MANDIA_INT8_BEGIN\',json.dumps(result,ensure_ascii=False),\'MANDIA_INT8_END\',flush=True)\n        return result\n    finally:\n        stage.DEVICE=old_device;torch.set_num_threads(threads)\nif __name__==\'__main__\':main()\n')
sys.path.insert(0,str(exports))
sys.modules.pop("quantifier_cpu",None)
try:
    from quantifier_cpu import main as compress_and_check
    int8_report=compress_and_check()
except Exception as error:
    import json
    (exports/"ERREUR-INT8.json").write_text(json.dumps({"error":repr(error),"original_checkpoints_unchanged":True}))
    print("EXPORT_INT8_ERREUR",repr(error),flush=True)


## Références et limites
[Attention Is All You Need](https://arxiv.org/abs/1706.03762) ; [GQA](https://arxiv.org/abs/2305.13245) ; [Efficient Architectures for Low-Resource Machine Translation](https://aclanthology.org/2025.lowresnlp-1.6/) ; [WAXAL](https://arxiv.org/abs/2602.02734).

Ces articles motivent les expériences ; leurs scores ne sont pas ceux de Mandia. Notre implémentation n'établit aucune nouveauté scientifique à elle seule. Évaluer plusieurs graines, un jeu conversationnel indépendant relu par des locuteurs, les variantes baoulé et le temps total sera nécessaire. Le corpus et les métriques automatiques comportent des limites documentées dans `PROVENANCE.json` et le rapport.

Code Mandia : Apache-2.0. Chaque corpus conserve sa licence et ses crédits. Ne pas distribuer les textes de tiers avec une nouvelle licence Mandia.
